In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║      NanoMind-SFT — Inference & Testing                            ║
# ║  Loads latest SFT checkpoint (local or HF) and runs test prompts   ║
# ╚══════════════════════════════════════════════════════════════════════╝

HF_TOKEN     = "api-key-goes-here"  # or set via HF_TOKEN env var
HF_SFT_REPO  = "shawneil/NanoMind-SFT-SmolTalk"
HF_BASE_REPO = "shawneil/NanoMind"        # fallback to base if no SFT ckpt yet

import os, sys, subprocess, time, textwrap
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "tiktoken", "huggingface_hub", "torch",
], check=True)

# ══════════════════════════════════════════════════════════════════════
# model.py  (inline — identical to training)
# ══════════════════════════════════════════════════════════════════════
MODEL_PY = r'''
import math, torch, torch.nn as nn, torch.nn.functional as F
from dataclasses import dataclass

@dataclass
class ModelConfig:
    vocab_size:    int   = 50257
    d_model:       int   = 512
    n_heads:       int   = 8
    n_kv_heads:    int   = 2
    n_layers:      int   = 8
    max_seq_len:   int   = 512
    ff_mult:       int   = 4
    dropout:       float = 0.0
    use_moe:       bool  = False
    num_experts:   int   = 4
    top_k_experts: int   = 2

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps    = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        x32 = x.float()
        rms = x32.pow(2).mean(-1, keepdim=True).add(self.eps).rsqrt()
        return (x32 * rms).to(x.dtype).clone() * self.weight

def precompute_freqs_cis(head_dim, max_len, theta=10000.0):
    freqs = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
    t     = torch.arange(max_len, device=freqs.device)
    freqs = torch.outer(t, freqs)
    return torch.polar(torch.ones_like(freqs), freqs)

def apply_rope(xq, xk, freqs_cis):
    def rot(x, f):
        xc  = torch.view_as_complex(x.float().reshape(*x.shape[:-1], -1, 2))
        out = torch.view_as_real(xc * f[:x.shape[1]].unsqueeze(0).unsqueeze(2)).flatten(3)
        return out.to(x.dtype)
    return rot(xq, freqs_cis), rot(xk, freqs_cis)

class GQAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.nh        = cfg.n_heads
        self.nkv       = cfg.n_kv_heads
        self.hd        = cfg.d_model // cfg.n_heads
        self.q         = nn.Linear(cfg.d_model, cfg.n_heads    * self.hd, bias=False)
        self.k         = nn.Linear(cfg.d_model, cfg.n_kv_heads * self.hd, bias=False)
        self.v         = nn.Linear(cfg.d_model, cfg.n_kv_heads * self.hd, bias=False)
        self.o         = nn.Linear(cfg.n_heads * self.hd, cfg.d_model,    bias=False)
        self.attn_drop = cfg.dropout
    def forward(self, x, freqs_cis):
        B, T, _ = x.shape
        q = self.q(x).view(B, T, self.nh,  self.hd)
        k = self.k(x).view(B, T, self.nkv, self.hd)
        v = self.v(x).view(B, T, self.nkv, self.hd)
        q, k = apply_rope(q, k, freqs_cis)
        reps  = self.nh // self.nkv
        k = k.repeat_interleave(reps, dim=2)
        v = v.repeat_interleave(reps, dim=2)
        q, k, v = q.transpose(1,2), k.transpose(1,2), v.transpose(1,2)
        out = F.scaled_dot_product_attention(
            q, k, v, attn_mask=None,
            dropout_p=self.attn_drop if self.training else 0.0,
            is_causal=True,
        )
        return self.o(out.transpose(1,2).contiguous().view(B, T, -1))

class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        h = int(cfg.d_model * cfg.ff_mult * 2 / 3)
        h = (h + 63) // 64 * 64
        self.w1 = nn.Linear(cfg.d_model, h, bias=False)
        self.w2 = nn.Linear(h, cfg.d_model, bias=False)
        self.w3 = nn.Linear(cfg.d_model, h, bias=False)
    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))

class MoEFF(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.top_k   = cfg.top_k_experts
        self.experts = nn.ModuleList([SwiGLU(cfg) for _ in range(cfg.num_experts)])
        self.gate    = nn.Linear(cfg.d_model, cfg.num_experts, bias=False)
    def forward(self, x):
        B, T, D = x.shape
        flat    = x.view(-1, D)
        w, idx  = torch.topk(F.softmax(self.gate(flat), dim=-1), self.top_k, dim=-1)
        out     = torch.zeros_like(flat)
        for i, expert in enumerate(self.experts):
            sel = (idx == i).any(dim=-1)
            if not sel.any(): continue
            rows = sel.nonzero(as_tuple=True)[0]
            kpos = (idx[rows] == i).nonzero(as_tuple=True)[1]
            out[rows] += w[rows, kpos].unsqueeze(-1) * expert(flat[rows])
        return out.view(B, T, D)

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.an   = RMSNorm(cfg.d_model)
        self.fn   = RMSNorm(cfg.d_model)
        self.attn = GQAttention(cfg)
        self.ff   = MoEFF(cfg) if cfg.use_moe else SwiGLU(cfg)
        self.drop = nn.Dropout(cfg.dropout)
    def forward(self, x, fc):
        x = x + self.drop(self.attn(self.an(x), fc))
        x = x + self.drop(self.ff(self.fn(x)))
        return x

class Sarvam(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg     = cfg
        self.embed   = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.drop    = nn.Dropout(cfg.dropout)
        self.blocks  = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.norm    = RMSNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.embed.weight = self.lm_head.weight
        self.register_buffer(
            "freqs_cis",
            precompute_freqs_cis(cfg.d_model // cfg.n_heads, cfg.max_seq_len * 2)
        )
        self.apply(self._init)
    def _init(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)
            if hasattr(m, "bias") and m.bias is not None:
                nn.init.zeros_(m.bias)
    def forward(self, idx, targets=None, loss_mask=None):
        B, T = idx.shape
        x    = self.drop(self.embed(idx))
        fc   = self.freqs_cis[:T]
        for blk in self.blocks:
            x = blk(x, fc)
        x      = self.norm(x)
        logits = self.lm_head(x)
        loss   = None
        if targets is not None:
            flat_logits  = logits.view(-1, logits.size(-1))
            flat_targets = targets.view(-1)
            if loss_mask is not None:
                flat_mask    = loss_mask.view(-1).bool()
                flat_logits  = flat_logits[flat_mask]
                flat_targets = flat_targets[flat_mask]
            loss = F.cross_entropy(flat_logits, flat_targets, ignore_index=-1)
        return logits, loss
    @torch.no_grad()
    def generate(self, idx, max_new_tokens=200, temperature=0.8, top_k=50,
                 top_p=0.95, repetition_penalty=1.1):
        generated = []
        for _ in range(max_new_tokens):
            ic     = idx[:, -self.cfg.max_seq_len:]
            logits, _ = self(ic)
            logits = logits[:, -1, :].float()

            # repetition penalty
            if repetition_penalty != 1.0 and generated:
                for token_id in set(generated):
                    if logits[0, token_id] > 0:
                        logits[0, token_id] /= repetition_penalty
                    else:
                        logits[0, token_id] *= repetition_penalty

            logits = logits / temperature

            # top-k
            if top_k > 0:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float("-inf")

            # top-p (nucleus)
            if top_p < 1.0:
                sorted_logits, sorted_idx = torch.sort(logits, descending=True)
                cum_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
                remove    = cum_probs - F.softmax(sorted_logits, dim=-1) > top_p
                sorted_logits[remove] = float("-inf")
                logits = torch.zeros_like(logits).scatter_(1, sorted_idx, sorted_logits)

            probs    = F.softmax(logits, dim=-1)
            next_tok = torch.multinomial(probs, 1)
            generated.append(next_tok.item())

            # stop at <|endoftext|>
            if next_tok.item() == self.cfg.vocab_size - 1:
                break

            idx = torch.cat([idx, next_tok], dim=1)
        return idx
    def num_params(self):
        return sum(p.numel() for p in self.parameters())
'''

# ══════════════════════════════════════════════════════════════════════
# inference.py
# ══════════════════════════════════════════════════════════════════════
INFER_PY = r'''
import os, sys, time, textwrap, torch
import tiktoken

# ── Prompt formatter (must match SFT training format exactly) ─────────
def make_prompt(instruction: str, input_text: str = "") -> str:
    if input_text.strip():
        return (
            f"### Instruction:\n{instruction.strip()}\n\n"
            f"### Input:\n{input_text.strip()}\n\n"
            f"### Response:\n"
        )
    return (
        f"### Instruction:\n{instruction.strip()}\n\n"
        f"### Response:\n"
    )


# ── Load model ────────────────────────────────────────────────────────
def load_model(ckpt_path, device):
    from model import ModelConfig, Sarvam
    ckpt = torch.load(ckpt_path, map_location=device)
    cfg_dict = ckpt.get("config", {})
    cfg = ModelConfig(**{k: v for k, v in cfg_dict.items()
                         if k in ModelConfig.__dataclass_fields__})
    model = Sarvam(cfg).to(device)
    model.load_state_dict(ckpt["model"], strict=True)
    model.eval()
    print(f"✅ Loaded checkpoint  (step={ckpt.get('step','?')} | "
          f"params={model.num_params()/1e6:.2f}M)", flush=True)
    return model, cfg


# ── Single inference call ─────────────────────────────────────────────
@torch.no_grad()
def run(model, enc, instruction, input_text="",
        max_new_tokens=256, temperature=0.7, top_k=50,
        top_p=0.9, repetition_penalty=1.15, device="cuda"):

    prompt     = make_prompt(instruction, input_text)
    prompt_ids = enc.encode_ordinary(prompt)
    idx        = torch.tensor([prompt_ids], dtype=torch.long, device=device)

    t0  = time.perf_counter()
    out = model.generate(
        idx,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_k=top_k,
        top_p=top_p,
        repetition_penalty=repetition_penalty,
    )
    dt = time.perf_counter() - t0

    # decode only the newly generated tokens
    new_ids  = out[0, len(prompt_ids):].tolist()
    response = enc.decode(new_ids)
    # strip trailing <|endoftext|> if present
    response = response.replace("<|endoftext|>", "").strip()
    tok_per_s = len(new_ids) / max(dt, 1e-6)

    return response, tok_per_s


# ── Pretty printer ────────────────────────────────────────────────────
SEP  = "─" * 70
SEP2 = "═" * 70

def print_result(instruction, response, input_text="", tok_per_s=0.0, idx=0):
    print(f"\n{SEP2}")
    print(f"  TEST {idx:02d}")
    print(SEP)
    print(f"  INSTRUCTION: {textwrap.fill(instruction, width=66, subsequent_indent='               ')}")
    if input_text.strip():
        print(f"  INPUT      : {textwrap.fill(input_text,   width=66, subsequent_indent='               ')}")
    print(SEP)
    print("  RESPONSE:")
    for line in response.splitlines():
        wrapped = textwrap.fill(line, width=68, initial_indent="    ",
                                subsequent_indent="    ")
        print(wrapped if wrapped.strip() else "")
    print(SEP)
    print(f"  ⚡ {tok_per_s:.1f} tok/s")
    print(SEP2)


# ══════════════════════════════════════════════════════════════════════
# TEST PROMPTS  —  covers the main Alpaca task categories
# ══════════════════════════════════════════════════════════════════════
TEST_CASES = [

    # ── 1. Basic factual question ─────────────────────────────────────
    {
        "instruction": "What is the capital of France?",
        "input": "",
        "tag": "factual",
    },

    # ── 2. Explanation / concept ──────────────────────────────────────
    {
        "instruction": "Explain what machine learning is in simple terms.",
        "input": "",
        "tag": "explanation",
    },

    # ── 3. Listing / enumeration ──────────────────────────────────────
    {
        "instruction": "Give three tips for staying healthy.",
        "input": "",
        "tag": "listing",
    },

    # ── 4. With input context ─────────────────────────────────────────
    {
        "instruction": "Classify the sentiment of the following sentence.",
        "input": "I absolutely loved the movie — the acting was superb!",
        "tag": "classification+input",
    },

    # ── 5. Text transformation ────────────────────────────────────────
    {
        "instruction": "Convert the following sentence to passive voice.",
        "input": "The chef cooked a delicious meal.",
        "tag": "transformation+input",
    },

    # ── 6. Creative writing ───────────────────────────────────────────
    {
        "instruction": "Write a short poem about the ocean at night.",
        "input": "",
        "tag": "creative",
    },

    # ── 7. Summarisation ─────────────────────────────────────────────
    {
        "instruction": "Summarize the following paragraph in one sentence.",
        "input": (
            "The Amazon rainforest, often referred to as the lungs of the Earth, "
            "produces about 20% of the world's oxygen and is home to millions of "
            "plant and animal species. Deforestation, driven by agriculture and "
            "logging, threatens this critical ecosystem at an alarming rate."
        ),
        "tag": "summarisation+input",
    },

    # ── 8. Brainstorming ─────────────────────────────────────────────
    {
        "instruction": "Brainstorm five unique business ideas for a college student.",
        "input": "",
        "tag": "brainstorming",
    },

    # ── 9. Rewriting / editing ────────────────────────────────────────
    {
        "instruction": "Rewrite the following sentence to make it more formal.",
        "input": "Hey, just wanted to let you know I can't make it to the meeting.",
        "tag": "editing+input",
    },

    # ── 10. Step-by-step reasoning ───────────────────────────────────
    {
        "instruction": "How do I make a cup of tea? Provide step-by-step instructions.",
        "input": "",
        "tag": "steps",
    },

    # ── 11. Comparison ───────────────────────────────────────────────
    {
        "instruction": "What are the differences between Python and JavaScript?",
        "input": "",
        "tag": "comparison",
    },

    # ── 12. Question answering with context ──────────────────────────
    {
        "instruction": "Answer the question based on the passage below.",
        "input": (
            "Photosynthesis is the process by which green plants convert sunlight "
            "into food using carbon dioxide and water, releasing oxygen as a byproduct.\n"
            "Question: What do plants release during photosynthesis?"
        ),
        "tag": "QA+input",
    },

    # ── 13. Open-ended advice ─────────────────────────────────────────
    {
        "instruction": "What advice would you give to someone starting to learn programming?",
        "input": "",
        "tag": "advice",
    },

    # ── 14. Definition ────────────────────────────────────────────────
    {
        "instruction": "Define the term 'inflation' in economics.",
        "input": "",
        "tag": "definition",
    },

    # ── 15. Stress test — longer output ──────────────────────────────
    {
        "instruction": (
            "Write a short story about a robot who learns to feel emotions "
            "for the first time."
        ),
        "input": "",
        "tag": "long_creative",
        "max_new_tokens": 400,
    },
]


# ══════════════════════════════════════════════════════════════════════
# Scoring helper  (simple heuristics — not a real eval, just sanity)
# ══════════════════════════════════════════════════════════════════════
def quick_score(response: str) -> dict:
    words     = response.split()
    sentences = [s.strip() for s in response.replace("?",".")
                 .replace("!",".").split(".") if s.strip()]
    return {
        "length_words":    len(words),
        "num_sentences":   len(sentences),
        "has_content":     len(words) > 5,
        "not_repetitive":  len(set(words)) / max(len(words), 1) > 0.3,
        "ends_cleanly":    not response.endswith(("###", "Instruction", "Response")),
    }


# ══════════════════════════════════════════════════════════════════════
# Runner
# ══════════════════════════════════════════════════════════════════════
def run_all_tests(ckpt_path, device="cuda",
                  temperature=0.7, top_k=50, top_p=0.9,
                  repetition_penalty=1.15):

    print(f"\n{'═'*70}")
    print("  NanoMind-SFT Inference Test Suite")
    print(f"  checkpoint : {ckpt_path}")
    print(f"  device     : {device}")
    print(f"  temp={temperature}  top_k={top_k}  top_p={top_p}  rep_pen={repetition_penalty}")
    print(f"{'═'*70}\n")

    model, cfg = load_model(ckpt_path, device)
    enc        = tiktoken.get_encoding("gpt2")

    scores = []
    for i, tc in enumerate(TEST_CASES, 1):
        max_new = tc.get("max_new_tokens", 256)
        response, tok_per_s = run(
            model, enc,
            instruction=tc["instruction"],
            input_text=tc.get("input", ""),
            max_new_tokens=max_new,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            repetition_penalty=repetition_penalty,
            device=device,
        )
        print_result(tc["instruction"], response,
                     input_text=tc.get("input",""),
                     tok_per_s=tok_per_s, idx=i)

        sc = quick_score(response)
        sc["tag"] = tc["tag"]
        sc["tok_per_s"] = round(tok_per_s, 1)
        scores.append(sc)

    # ── Summary table ─────────────────────────────────────────────────
    print(f"\n{'═'*70}")
    print("  SUMMARY")
    print(f"  {'#':>3}  {'tag':<24}  {'words':>5}  {'clean':>5}  {'diverse':>7}  {'tok/s':>6}")
    print(f"  {'─'*3}  {'─'*24}  {'─'*5}  {'─'*5}  {'─'*7}  {'─'*6}")
    pass_count = 0
    for i, (tc, sc) in enumerate(zip(TEST_CASES, scores), 1):
        ok = sc["has_content"] and sc["not_repetitive"] and sc["ends_cleanly"]
        if ok: pass_count += 1
        flag = "✓" if ok else "✗"
        print(f"  {flag}{i:>2}  {sc['tag']:<24}  {sc['length_words']:>5}  "
              f"{'yes' if sc['ends_cleanly'] else 'no':>5}  "
              f"{'yes' if sc['not_repetitive'] else 'no':>7}  "
              f"{sc['tok_per_s']:>6}")
    print(f"  {'─'*55}")
    print(f"  Passed: {pass_count}/{len(TEST_CASES)}")
    print(f"{'═'*70}\n")


# ══════════════════════════════════════════════════════════════════════
# Entry point
# ══════════════════════════════════════════════════════════════════════
if __name__ == "__main__":
    import argparse
    p = argparse.ArgumentParser()
    p.add_argument("--ckpt",               type=str,   default=None)
    p.add_argument("--use_base",           action="store_true",
                   help="Load base pretrained model instead of SFT")
    p.add_argument("--hf_sft_repo",        type=str,   default="shawneil/NanoMind-SFT")
    p.add_argument("--hf_base_repo",       type=str,   default="shawneil/NanoMind")
    p.add_argument("--hf_token",           type=str,   default=None)
    p.add_argument("--temperature",        type=float, default=0.7)
    p.add_argument("--top_k",              type=int,   default=50)
    p.add_argument("--top_p",              type=float, default=0.9)
    p.add_argument("--repetition_penalty", type=float, default=1.15)
    p.add_argument("--device",             type=str,
                   default="cuda" if torch.cuda.is_available() else "cpu")
    p.add_argument("--interactive",        action="store_true",
                   help="Drop into interactive prompt after tests")
    args = p.parse_args()

    hf_token = args.hf_token or os.environ.get("HF_TOKEN", "")
    repo     = args.hf_base_repo if args.use_base else args.hf_sft_repo

    ckpt_path = args.ckpt
    if not ckpt_path:
        # auto-find local checkpoint
        import glob
        from pathlib import Path
        pattern  = "checkpoints/checkpoint_step_*.pt" if args.use_base \
                   else "sft_checkpoints/sft_step_*.pt"
        local    = sorted(glob.glob(f"/kaggle/working/{pattern}"),
                          key=lambda x: int(x.split("_")[-1].replace(".pt","")))
        if local:
            ckpt_path = local[-1]
            print(f"[AUTO] Using local: {Path(ckpt_path).name}")
        else:
            # download from HF
            print(f"[AUTO] Downloading latest from {repo} ...")
            from huggingface_hub import hf_hub_download, list_repo_files
            prefix = "smoltalk_step_" if args.use_base else "smoltalk_step_"
            files  = [f for f in list_repo_files(repo, token=hf_token)
                      if f.startswith(prefix) and f.endswith(".pt")]
            latest = sorted(files, key=lambda x: int(x.split("_")[-1].replace(".pt","")))[-1]
            out_dir = ("/kaggle/working/checkpoints" if args.use_base
                       else "/kaggle/working/sft_checkpoints")
            os.makedirs(out_dir, exist_ok=True)
            ckpt_path = hf_hub_download(repo_id=repo, filename=latest,
                                        token=hf_token, local_dir=out_dir)
            print(f"[AUTO] Downloaded: {latest}")

    # write model.py if not present
    model_dest = "/kaggle/working/model.py"
    if not os.path.isfile(model_dest):
        with open(model_dest, "w") as f:
            f.write(MODEL_PY)
    sys.path.insert(0, "/kaggle/working")

    # ── run test suite ────────────────────────────────────────────────
    run_all_tests(
        ckpt_path=ckpt_path,
        device=args.device,
        temperature=args.temperature,
        top_k=args.top_k,
        top_p=args.top_p,
        repetition_penalty=args.repetition_penalty,
    )

    # ── optional interactive mode ─────────────────────────────────────
    if args.interactive:
        import tiktoken
        from model import ModelConfig, Sarvam
        enc   = tiktoken.get_encoding("gpt2")
        model, _ = load_model(ckpt_path, args.device)
        print("\n" + "═"*70)
        print("  INTERACTIVE MODE  (type 'quit' to exit)")
        print("  Leave 'Input' blank if not needed.")
        print("═"*70)
        while True:
            instruction = input("\n📝 Instruction: ").strip()
            if instruction.lower() in ("quit", "exit", "q"): break
            input_text  = input("📎 Input (optional): ").strip()
            temp        = input(f"🌡  Temperature [{args.temperature}]: ").strip()
            temp        = float(temp) if temp else args.temperature
            response, tps = run(
                model, enc, instruction, input_text,
                temperature=temp, top_k=args.top_k, top_p=args.top_p,
                repetition_penalty=args.repetition_penalty,
                device=args.device,
            )
            print(f"\n{'─'*70}\n🤖 Response:\n{response}\n{'─'*70}")
            print(f"⚡ {tps:.1f} tok/s")
'''

# ── Write files ────────────────────────────────────────────────────────
os.makedirs("/kaggle/working", exist_ok=True)
with open("/kaggle/working/model.py",     "w") as f: f.write(MODEL_PY)
with open("/kaggle/working/inference.py", "w") as f: f.write(INFER_PY)
print("✅  model.py + inference.py written to /kaggle/working/")

# ══════════════════════════════════════════════════════════════════════
# Run tests
# ══════════════════════════════════════════════════════════════════════
import subprocess, sys

CMD = [
    sys.executable, "/kaggle/working/inference.py",
    "--hf_sft_repo",  HF_SFT_REPO,
    "--hf_base_repo", HF_BASE_REPO,
    "--hf_token",     HF_TOKEN,
    "--temperature",  "0.7",
    "--top_k",        "50",
    "--top_p",        "0.9",
    "--repetition_penalty", "1.15",
    # "--use_base",    # ← uncomment to test base model instead of SFT
    # "--interactive", # ← uncomment to drop into chat after tests
]

ret = subprocess.run(CMD)
if ret.returncode != 0:
    raise RuntimeError("Inference script failed — check output above")

✅  model.py + inference.py written to /kaggle/working/
[AUTO] Downloading latest from shawneil/NanoMind-SFT-SmolTalk ...
[AUTO] Downloaded: smoltalk_step_21021.pt

══════════════════════════════════════════════════════════════════════
  NanoMind-SFT Inference Test Suite
  checkpoint : /kaggle/working/sft_checkpoints/smoltalk_step_21021.pt
  device     : cuda
  temp=0.7  top_k=50  top_p=0.9  rep_pen=1.15
══════════════════════════════════════════════════════════════════════

✅ Loaded checkpoint  (step=21021 | params=48.28M)

══════════════════════════════════════════════════════════════════════
  TEST 01
──────────────────────────────────────────────────────────────────────
  INSTRUCTION: What is the capital of France?
──────────────────────────────────────────────────────────────────────
  RESPONSE:
    The capital of France is Paris. It is a city known for its rich
    history, culture, and iconic landmarks. The most notable example
    of this type of urban area lies in the historic 

In [3]:
# ── Quick test cell (run after inference.py is already written) ────────
import sys
sys.path.insert(0, "/kaggle/working")

from inference import load_model, run, run_all_tests, TEST_CASES
import tiktoken

# Change this to your actual latest checkpoint path
# or set to None to auto-download from HF
CKPT = None   # e.g. "/kaggle/working/sft_checkpoints/sft_step_1500.pt"

enc = tiktoken.get_encoding("gpt2")

# ── Auto-find latest local SFT checkpoint ─────────────────────────────
import glob
from pathlib import Path

local = sorted(
    glob.glob("/kaggle/working/sft_checkpoints/smoltalk_step_*.pt"),
    key=lambda x: int(x.split("_")[-1].replace(".pt", ""))
)
if local:
    CKPT = local[-1]
    print(f"Using: {Path(CKPT).name}")
else:
    print("No local SFT ckpt found — will download from HF")

# ── Load model ────────────────────────────────────────────────────────
model, cfg = load_model(CKPT or "auto", device="cuda")

# ══════════════════════════════════════════════════════════════════════
# Option A — Run full test suite (all 15 prompts)
# ══════════════════════════════════════════════════════════════════════
run_all_tests(CKPT, device="cuda", temperature=0.7, top_k=50,
              top_p=0.9, repetition_penalty=1.15)

# ══════════════════════════════════════════════════════════════════════
# Option B — Single custom prompt
# ══════════════════════════════════════════════════════════════════════
response, tps = run(
    model, enc,
    instruction="Explain what photosynthesis is in simple terms.",
    input_text="",          # leave blank if no input context needed
    max_new_tokens=200,
    temperature=0.7,
    top_k=50,
    top_p=0.9,
    repetition_penalty=1.15,
    device="cuda",
)
print(f"\nResponse:\n{response}")
print(f"\n⚡ {tps:.1f} tok/s")

Using: smoltalk_step_21021.pt
✅ Loaded checkpoint  (step=21021 | params=48.28M)

══════════════════════════════════════════════════════════════════════
  NanoMind-SFT Inference Test Suite
  checkpoint : /kaggle/working/sft_checkpoints/smoltalk_step_21021.pt
  device     : cuda
  temp=0.7  top_k=50  top_p=0.9  rep_pen=1.15
══════════════════════════════════════════════════════════════════════

✅ Loaded checkpoint  (step=21021 | params=48.28M)

══════════════════════════════════════════════════════════════════════
  TEST 01
──────────────────────────────────────────────────────────────────────
  INSTRUCTION: What is the capital of France?
──────────────────────────────────────────────────────────────────────
  RESPONSE:
    The capital of France is Paris. It has a large, open-air public
    square with an area of approximately 40% (4th and 5th streets)
    for all purposes. The city also hosts the famous paralegal
    society, which was founded in 1807.
──────────────────────────────────